<a href="https://colab.research.google.com/github/Merc-Gonsalves/Health-Project/blob/main/test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q monai kagglehub xgboost shap grad-cam

In [ ]:
import os
import random
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms

import monai
from monai.networks.nets import DenseNet121

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, roc_curve, precision_recall_curve,
)

import xgboost as xgb
import shap

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device         : {device}')
print(f'torch          : {torch.__version__}')
print(f'monai          : {monai.__version__}')
print(f'xgboost        : {xgb.__version__}')

In [ ]:
import kagglehub

path = kagglehub.dataset_download('ajithdari/multi-modal-breast-cancer-dataset')
DATA_PATH = Path(path)
print(f'Dataset root: {DATA_PATH}')

# Quick look at what we got
for p in sorted(DATA_PATH.rglob('*'))[:25]:
    if p.is_dir():
        print(f'  [d] {p.relative_to(DATA_PATH)}')
    else:
        print(f'  [f] {p.relative_to(DATA_PATH)}  ({p.stat().st_size:,} B)')